# Speaker study, Phase 3 + 4: label-swap experiment (Qwen 2.5 7B primary, Gemma 2 2B secondary)

Runs all 1,260 stimuli (420 transcripts x final label `[Assistant]:` / `[User]:` / `[Moderator]:`)
through **Qwen 2.5 7B base** (layer 8, first 9 blocks, bf16), then **Gemma 2 2B base** (layer 7),
and runs the preregistered analysis on each.

**Integrity check:** before any forward pass, the notebook verifies that the scripts it writes and the
rebuilt stimuli match the SHA-256 hashes frozen in `PREREGISTRATION.md`. If the check fails, stop and
tell Claude; do not edit the cells.

**How to run:** T4 GPU runtime, then Runtime -> Run all. About 45-60 min. If the runtime disconnects
after Qwen has finished, rerun the setup cells (1-4 and the script cells), the integrity cell and the
Gemma cell. New runs never overwrite earlier ones. The last cell downloads `speaker_study_phase3.zip`
(also saved to `MyDrive/speaker_study/`); attach it in the Claude session.

In [ ]:
# 1. GPU check. Runtime -> Change runtime type -> T4 GPU (free tier).
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU first."

In [ ]:
# 2. Mount Google Drive; all outputs go to MyDrive/speaker_study/ (never overwritten).
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = "/content/drive/MyDrive/speaker_study"
import os; os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# 3. Hugging Face token from Colab secrets (key icon on the left, name HF_TOKEN, notebook access on).
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
# 4. Clone the paper's repo at the pinned commit (read-only use; nothing in it is modified).
!rm -rf /content/Pain-axis
!git clone -q https://github.com/valen-research/Pain-axis /content/Pain-axis
!git -C /content/Pain-axis checkout -q 7c256502ed3d98e4e6379290fe7db2f93cb8d025
!git -C /content/Pain-axis log -1 --format='%H %s'
# Uses Colab's preinstalled torch/transformers/accelerate/pandas (versions are logged to env.txt).
!mkdir -p /content/speaker_study_scripts

In [ ]:
%%writefile /content/speaker_study_scripts/pa_common.py
"""Shared helpers for the speaker study: dataset and vector loading, model loading, and
final-token readout. The format, readout and projection logic is copied from
Pain-axis scripts/4.1_self_other/01_screen_scenarios.py (commit 7c25650) so that our
projections are comparable with the shipped ones.
"""

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

PAIN_AXIS_COMMIT = "7c256502ed3d98e4e6379290fe7db2f93cb8d025"
SEED = 0

# Dataset stratum -> our group name.
STRATUM_TO_GROUP = {
    "self_directed": "harm_to_model",
    "vicarious_empathic": "user_suffering",
    "neutral_filler": "neutral",
}

# Same keys and order as the repo's 4.1 screen.
VECTOR_KEYS = [
    "s1_pain_vector", "s2_pain_vector",
    "fear_vector", "negemotion_vector", "negworld_vector",
    "bodysens_vector", "arousal_vector", "random_vector", "numb_vector", "sadness_vector",
]

# Base models from the spec: HF repo -> repo's model name (used in vector/result file names).
MODELS = {
    "google/gemma-2-2b": "Gemma_2_2B_base",
    "Qwen/Qwen2.5-7B": "Qwen_2.5_7B_base",
    "meta-llama/Llama-3.1-8B": "Llama_3.1_8B_base",
    "google/gemma-2-9b": "Gemma_2_9B_base",
}

DTYPES = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}


def load_scenarios(pain_axis_dir):
    path = Path(pain_axis_dir) / "datasets" / "4.1_self_other_420_scenarios.json"
    with open(path, encoding="utf-8") as f:
        items = json.load(f)
    for it in items:
        it["group"] = STRATUM_TO_GROUP[it["stratum"]]
        it["n_user_turns"] = sum(line.startswith("[User]:") for line in it["text"].split("\n"))
    return items


def load_vectors(pain_axis_dir, model_name):
    """Unit vectors from the file the 4.1 screen uses, and its layer."""
    path = Path(pain_axis_dir) / "results" / "vectors_full_steering" / f"vectors_full_{model_name}.pt"
    data = torch.load(path, map_location="cpu", weights_only=False)
    units = {}
    for k in VECTOR_KEYS:
        if data.get(k) is not None:
            v = data[k].float().numpy()
            n = np.linalg.norm(v)
            units[k] = v / n if n > 0 else v
    return int(data["layer"]), units


def load_shipped_screen(pain_axis_dir, model_name):
    import pandas as pd
    path = Path(pain_axis_dir) / "results" / "4.1_self_other" / "per_model" / f"screen_v2_{model_name}.csv"
    return pd.read_csv(path)


def load_model(repo, dtype="bf16", attn="default", keep_layers=None):
    """Load tokenizer and model on the GPU.

    keep_layers: if set, build the model with only its first `keep_layers` decoder blocks.
    The output of block L depends only on blocks 0..L, so the readout is unchanged; this
    lets 7-8B models fit on a 16 GB GPU in bf16. Phase 1 verifies the equivalence.
    """
    import transformers
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(repo)
    kwargs = {"low_cpu_mem_usage": True, "device_map": "cuda" if torch.cuda.is_available() else "cpu"}
    # Newer transformers renamed torch_dtype -> dtype; an unknown kwarg would silently be
    # written into the config, so pick by version and assert the dtype afterwards.
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    kwargs["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = DTYPES[dtype]
    if attn != "default":
        kwargs["attn_implementation"] = attn
    if keep_layers is not None:
        cfg = AutoConfig.from_pretrained(repo)
        cfg.num_hidden_layers = keep_layers
        if isinstance(getattr(cfg, "layer_types", None), list):
            cfg.layer_types = cfg.layer_types[:keep_layers]
        kwargs["config"] = cfg
    model = AutoModelForCausalLM.from_pretrained(repo, **kwargs)
    model.eval()
    got = next(model.parameters()).dtype
    assert got == DTYPES[dtype], f"model loaded as {got}, expected {DTYPES[dtype]}"
    return tok, model


def decoder_layers(model):
    return model.model.layers if hasattr(model.model, "layers") else model.model.language_model.layers


def encode(tok, text, device):
    """As the repo does for base models: default special tokens (adds BOS where the tokenizer does)."""
    return tok(text, return_tensors="pt").input_ids.to(device)


class FinalTokenReader:
    """Forward hook on decoder block `layer`; captures the final-token output in fp32."""

    def __init__(self, model, layer):
        self.model, self.act = model, None
        self.handle = decoder_layers(model)[layer].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        self.act = hs[0, -1, :].float().cpu().numpy()

    @torch.no_grad()
    def __call__(self, input_ids):
        self.model(input_ids=input_ids)
        return self.act

    def remove(self):
        self.handle.remove()


def git_head(path):
    try:
        return subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "unknown"


def env_info():
    import transformers
    info = {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    if torch.cuda.is_available():
        info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    return info


def pip_freeze(path):
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    Path(path).write_text(out)


def zscore(x, mean, sd):
    return (np.asarray(x) - mean) / (sd + 1e-8)

In [ ]:
%%writefile /content/speaker_study_scripts/02_build_stimuli.py
"""Phase 2: build the label-swapped stimuli and verify them.

For each of the 420 base-model transcripts, writes three variants that are byte-identical
except for the final speaker label:
  assistant_next   ...\\n[Assistant]:   (original)
  user_next        ...\\n[User]:
  moderator_next   ...\\n[Moderator]:   (third label, fixed before any Phase 3 run)

Text checks (always): each variant equals the shared prefix plus its label, the prefix
ends with '\\n', and the variants differ only after the prefix.

Token checks (with --tokenizers, needs Hugging Face access): tokenizes every variant,
finds the common token prefix across the three conditions, and records the differing
tail tokens and the final token of each condition. The tail may be shorter than the label
(e.g. if '\\n[' is one token shared by all labels); an item is flagged only if its tail
is not a non-empty suffix of the label, i.e. if tokens outside the label differ.

Outputs (never overwritten):
  <out-dir>/stimuli.jsonl                       one row per scenario x condition
  <out-dir>/token_check_<model>.csv / .json     per-tokenizer checks (if --tokenizers)
"""

import argparse
import json
from pathlib import Path

import pa_common as pc

LABELS = {"assistant_next": "[Assistant]:", "user_next": "[User]:", "moderator_next": "[Moderator]:"}
ORIG = LABELS["assistant_next"]


def build(items):
    rows = []
    for it in items:
        text = it["text"]
        assert text.endswith("\n" + ORIG), it["id"]
        prefix = text[: -len(ORIG)]
        for cond, label in LABELS.items():
            v = prefix + label
            assert v[: len(prefix)] == prefix and v[len(prefix):] == label
            rows.append({"scenario_id": it["id"], "category": it["category"], "group": it["group"],
                         "n_user_turns": it["n_user_turns"], "condition": cond, "label": label, "text": v})
        assert rows[-3]["text"] == text  # assistant_next is the original, byte for byte
    return rows


def token_check(rows, repo):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(repo)
    by_id = {}
    for r in rows:
        by_id.setdefault(r["scenario_id"], {})[r["condition"]] = r
    out, problems = [], []
    for sid, conds in by_id.items():
        ids = {c: tok(conds[c]["text"]).input_ids for c in LABELS}
        n = min(len(v) for v in ids.values())
        k = 0
        while k < n and len({tuple(v[: k + 1]) for v in ids.values()}) == 1:
            k += 1
        rec = {"scenario_id": sid, "n_common_tokens": k}
        for c, v in ids.items():
            tail = v[k:]
            rec[f"{c}_n_tokens"] = len(v)
            rec[f"{c}_tail_ids"] = " ".join(map(str, tail))
            rec[f"{c}_tail"] = tok.decode(tail)
            rec[f"{c}_final_id"] = v[-1]
            rec[f"{c}_final"] = tok.decode(v[-1:])
            if not tail or not LABELS[c].endswith(tok.decode(tail)):
                problems.append((sid, c, tok.decode(tail)))
        out.append(rec)
    return out, problems


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pain-axis-dir", required=True)
    p.add_argument("--out-dir", required=True)
    p.add_argument("--tokenizers", nargs="*", default=[], choices=sorted(pc.MODELS))
    args = p.parse_args()

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows = build(pc.load_scenarios(args.pain_axis_dir))
    stim = out / "stimuli.jsonl"
    if stim.exists():
        existing = [json.loads(line) for line in stim.read_text().splitlines()]
        assert existing == rows, f"{stim} exists with different content; refusing to overwrite"
        print(f"{stim} already exists and matches")
    else:
        stim.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
        print(f"wrote {stim}: {len(rows)} rows")

    import pandas as pd
    for repo in args.tokenizers:
        name = pc.MODELS[repo]
        csv_path, json_path = out / f"token_check_{name}.csv", out / f"token_check_{name}.json"
        if csv_path.exists():
            print(f"{csv_path} exists; skipping")
            continue
        recs, problems = token_check(rows, repo)
        df = pd.DataFrame(recs)
        df.to_csv(csv_path, index=False)
        summary = {"model": name, "repo": repo, "n_scenarios": len(df),
                   "n_tail_mismatches": len(problems), "tail_mismatch_examples": problems[:10]}
        for c in LABELS:
            summary[f"{c}_tails"] = df[f"{c}_tail_ids"].value_counts().to_dict()
            summary[f"{c}_final_tokens"] = (df[f"{c}_final"] + " (" + df[f"{c}_final_id"].astype(str) + ")").value_counts().to_dict()
        summary["same_final_token_all_conditions"] = bool(
            (df["assistant_next_final_id"] == df["user_next_final_id"]).all()
            and (df["assistant_next_final_id"] == df["moderator_next_final_id"]).all())
        json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
        print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/speaker_study_scripts/03_run_conditions.py
"""Phase 3: run the label-swapped stimuli and save per-item projections.

For every scenario x condition in stimuli.jsonl (assistant_next, user_next,
moderator_next), reads the final-token output of the model's 4.1 steering-layer block
(same readout as Phase 1) and saves raw projections onto every shipped direction,
cosine similarities, and the activation norm.

Fixed z-scoring (preregistered): per vector, mean and population SD come from the 420
assistant_next items of this run (= the original 4.1 transcripts) and are applied
unchanged to every condition. The combined pain axis is
    proj_pain = (z_S1 + z_S2) / 2      with those fixed statistics,
which is the paper's definition, so label effects d = proj_pain(cond) - proj_pain(assistant_next)
depend only on raw projection differences scaled by the fixed SDs.

Sanity check: assistant_next projections are compared with the shipped 4.1 screen
(must match as in Phase 1).

Output (never overwritten): <out-root>/results/<model>/phase3_<run-id>/items.csv,
pool_stats.json, run_info.json.
"""

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import pa_common as pc

SHORT = {"s1_pain_vector": "S1", "s2_pain_vector": "S2", "fear_vector": "fear",
         "negemotion_vector": "negemo", "sadness_vector": "sadness", "negworld_vector": "negworld",
         "bodysens_vector": "bodysens", "arousal_vector": "arousal", "random_vector": "random",
         "numb_vector": "numb"}


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pain-axis-dir", required=True)
    p.add_argument("--stimuli", required=True, help="stimuli.jsonl from 02_build_stimuli.py")
    p.add_argument("--out-root", required=True)
    p.add_argument("--model-repo", default="Qwen/Qwen2.5-7B", choices=sorted(pc.MODELS))
    p.add_argument("--dtype", default="bf16", choices=sorted(pc.DTYPES))
    p.add_argument("--attn", default="default", choices=["default", "eager", "sdpa"])
    p.add_argument("--no-truncate", action="store_true", help="load all blocks (default: first layer+1 blocks)")
    p.add_argument("--run-id", default=None)
    args = p.parse_args()

    np.random.seed(pc.SEED)
    torch.manual_seed(pc.SEED)
    model_name = pc.MODELS[args.model_repo]
    env = pc.env_info()
    run_id = args.run_id or time.strftime("%Y%m%d_%H%M%S") + "_" + (env["gpu"] or "cpu").replace(" ", "")
    out = Path(args.out_root) / "results" / model_name / f"phase3_{run_id}"
    if out.exists():
        raise SystemExit(f"{out} exists; refusing to overwrite")
    out.mkdir(parents=True)
    pc.pip_freeze(out / "env.txt")

    stimuli = [json.loads(line) for line in Path(args.stimuli).read_text().splitlines()]
    layer, units = pc.load_vectors(args.pain_axis_dir, model_name)
    tok, model = pc.load_model(args.model_repo, args.dtype, args.attn,
                               keep_layers=None if args.no_truncate else layer + 1)
    reader = pc.FinalTokenReader(model, layer)
    print(f"{model_name}: layer {layer}, {len(pc.decoder_layers(model))} blocks loaded, {len(stimuli)} stimuli")

    rows, t0 = [], time.time()
    for i, s in enumerate(stimuli):
        ids = pc.encode(tok, s["text"], model.device)
        act = reader(ids)
        norm = float(np.linalg.norm(act))
        row = {"model": model_name, "scenario_id": s["scenario_id"], "category": s["category"],
               "group": s["group"], "n_user_turns": s["n_user_turns"], "condition": s["condition"],
               "n_tokens": ids.shape[1], "final_token": tok.decode(ids[0, -1:]),
               "final_token_id": int(ids[0, -1]), "act_norm": norm}
        for k, u in units.items():
            proj = float(np.dot(act, u))
            row[f"proj_{SHORT[k]}"] = proj
            row[f"cos_{SHORT[k]}"] = proj / norm if norm > 0 else 0.0
        rows.append(row)
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(stimuli)} ({time.time() - t0:.0f}s)")
    reader.remove()
    df = pd.DataFrame(rows)

    # Fixed pool statistics from the original (assistant_next) transcripts.
    base = df[df["condition"] == "assistant_next"]
    assert len(base) == 420
    pool = {}
    for k in units:
        col = f"proj_{SHORT[k]}"
        m, sd = float(base[col].mean()), float(base[col].std(ddof=0))
        pool[SHORT[k]] = {"mean": m, "sd": sd}
        df[f"z_{SHORT[k]}"] = pc.zscore(df[col], m, sd)
    df["proj_pain"] = (df["z_S1"] + df["z_S2"]) / 2
    # Cosine-based pain axis (norm control): cosines z-scored with fixed assistant_next stats.
    for s in ("S1", "S2"):
        c = df.loc[df["condition"] == "assistant_next", f"cos_{s}"]
        df[f"zcos_{s}"] = pc.zscore(df[f"cos_{s}"], float(c.mean()), float(c.std(ddof=0)))
    df["cos_pain"] = (df["zcos_S1"] + df["zcos_S2"]) / 2

    # Sanity: assistant_next vs shipped 4.1 screen.
    sh = pc.load_shipped_screen(args.pain_axis_dir, model_name).set_index("id")
    b = df[df["condition"] == "assistant_next"].set_index("scenario_id")
    check = {}
    for k in ("s1_pain_vector", "s2_pain_vector"):
        a, s = b[f"proj_{SHORT[k]}"], sh.loc[b.index, f"{k}_proj"]
        check[SHORT[k]] = {"r": float(np.corrcoef(a, s)[0, 1]), "max_abs_diff": float((a - s).abs().max())}
    print("assistant_next vs shipped:", check)

    df.to_csv(out / "items.csv", index=False)
    (out / "pool_stats.json").write_text(json.dumps(pool, indent=2))
    info = {"phase": 3, "model": model_name, "model_repo": args.model_repo, "run_id": run_id, "layer": layer,
            "dtype": args.dtype, "attn_implementation": getattr(model.config, "_attn_implementation", "unknown"),
            "blocks_loaded": len(pc.decoder_layers(model)), "pain_axis_commit": pc.git_head(args.pain_axis_dir),
            "env": env, "n_rows": len(df), "assistant_next_vs_shipped": check,
            "final_tokens": df.groupby(["condition", "final_token_id", "final_token"]).size()
                              .reset_index(name="n").to_dict(orient="records")}
    (out / "run_info.json").write_text(json.dumps(info, indent=2))
    print(json.dumps(info["final_tokens"], indent=1))
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/speaker_study_scripts/04_analyze.py
"""Phase 4: preregistered analysis of the label-swap experiment.

Input: items.csv from 03_run_conditions.py (one row per scenario x condition).

For each scenario i and swapped condition c (user_next, moderator_next):
    d_i = y_i(c) - y_i(assistant_next)
Contrasts (group means are item means; categories within a group have equal n):
    Delta_harm   = mean(d | harm_to_model)  - mean(d | neutral)     H-speaker: < 0
    Delta_suffer = mean(d | user_suffering) - mean(d | neutral)     H-speaker: > 0
    I            = Delta_suffer - Delta_harm                         H-speaker: > 0, H-model: ~ 0
Uncertainty: stratified bootstrap (resample scenarios within category, B = 10,000,
seed 0; percentile 95% CIs, plus the 90% CI of I for the equivalence test) and a
permutation test on I (shuffle group labels of the d_i, 10,000 permutations,
two-sided p = (1 + #{|I_perm| >= |I_obs|}) / (1 + n_perm)).

Category-level inference (category_level): the same three contrasts on the 21 category
means, with exact permutation tests over category labels and Welch t intervals. Valid
if the label effect varies by category; the item-level tests above are not (see
tools/simulate_validation.py). Both verdicts are computed; PREREGISTRATION.md fixes
how they combine (co-primary, conjunctive: see combine()).

Primary outcome: proj_pain (paper's pain axis, mean of S1 and S2 z with fixed
assistant_next statistics), condition user_next. The verdict rule is in classify().
Everything else is secondary or descriptive.

Outputs (new folder, never overwritten): summary.json, contrasts.csv, per_category.csv,
levels.csv, REPORT_analysis.md, fig1-fig4 (PNG + PDF).
"""

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

GROUPS = ["harm_to_model", "user_suffering", "neutral"]
GROUP_LABEL = {"harm_to_model": "Harm directed at the model", "user_suffering": "User suffering",
               "neutral": "Neutral controls"}
CONDS = ["assistant_next", "user_next", "moderator_next"]
COND_LABEL = {"assistant_next": "[Assistant]:", "user_next": "[User]:", "moderator_next": "[Moderator]:"}
COND_SHORT = {"assistant_next": "Assistant", "user_next": "User", "moderator_next": "Moderator"}
PRIMARY = "proj_pain"
# (column, description, role)
OUTCOMES = [
    ("proj_pain", "pain axis, mean of S1/S2 z (fixed assistant_next stats)", "primary"),
    ("z_S1", "S1 pain vector, fixed z", "secondary"),
    ("z_S2", "S2 pain vector, fixed z", "secondary"),
    ("proj_S1", "S1 pain vector, raw projection", "secondary"),
    ("proj_S2", "S2 pain vector, raw projection", "secondary"),
    ("z_fear", "fear, fixed z", "secondary"),
    ("z_negemo", "negative emotion, fixed z", "secondary"),
    ("z_sadness", "sadness, fixed z", "secondary"),
    ("cos_pain", "pain axis from cosine similarities (norm control), fixed z", "secondary"),
    ("act_norm", "residual norm at the readout token", "secondary"),
]
# Series colors: first three slots of the reference categorical palette, validated
# all-pairs (light). Aqua is < 3:1 on the surface, so every series also gets a marker
# shape and a direct label.
COLORS = {"harm_to_model": "#2a78d6", "user_suffering": "#eb6834", "neutral": "#1baf7a"}
MARKERS = {"harm_to_model": "o", "user_suffering": "s", "neutral": "^"}
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"


# ---------------------------------------------------------------- statistics -----
class Design:
    """Items in a fixed order, plus a bootstrap index matrix that resamples within category."""

    def __init__(self, meta, n_boot, rng):
        self.meta = meta.reset_index(drop=True)
        self.groups = self.meta["group"].to_numpy()
        cats = self.meta["category"].to_numpy()
        blocks = []
        for c in pd.unique(cats):
            idx = np.flatnonzero(cats == c)
            blocks.append(idx[rng.integers(0, len(idx), size=(n_boot, len(idx)))])
        # Columns are ordered category by category; reorder so column j resamples item j's category.
        order = np.concatenate([np.flatnonzero(cats == c) for c in pd.unique(cats)])
        idx_mat = np.concatenate(blocks, axis=1)
        self.boot = np.empty_like(idx_mat)
        self.boot[:, order] = idx_mat

    def mask(self, g):
        return self.groups == g


def contrasts(v, design):
    """v: (..., N) values aligned with design.meta. Returns group means and contrasts."""
    m = {g: v[..., design.mask(g)].mean(axis=-1) for g in GROUPS if design.mask(g).any()}
    out = {f"mean_{g}": m[g] for g in m}
    if "neutral" in m:
        out["delta_harm"] = m["harm_to_model"] - m["neutral"]
        out["delta_suffer"] = m["user_suffering"] - m["neutral"]
    out["I"] = m["user_suffering"] - m["harm_to_model"]  # = delta_suffer - delta_harm
    return out


def boot_summary(v, design):
    obs = contrasts(v, design)
    bs = contrasts(v[design.boot], design)
    res = {}
    for k, val in obs.items():
        b = bs[k]
        res[k] = {"est": float(val), "ci95": [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]}
        if k == "I":
            res[k]["ci90"] = [float(np.percentile(b, 5)), float(np.percentile(b, 95))]
    return res


def perm_test_I(d, groups, n_perm, rng):
    obs = d[groups == "user_suffering"].mean() - d[groups == "harm_to_model"].mean()
    perms = np.argsort(rng.random((n_perm, len(d))), axis=1)
    g = groups[perms]
    s, h = g == "user_suffering", g == "harm_to_model"
    i_perm = (d * s).sum(1) / s.sum(1) - (d * h).sum(1) / h.sum(1)
    p = (1 + np.sum(np.abs(i_perm) >= abs(obs))) / (1 + n_perm)
    return float(obs), float(p)


def _exact_perm(a, b):
    """Two-sided exact permutation p for mean(a) - mean(b), over all splits of the pooled values."""
    from itertools import combinations
    pooled = np.concatenate([a, b])
    n, na, total = len(pooled), len(a), pooled.sum()
    obs = a.mean() - b.mean()
    stats = np.array([pooled[list(s)].sum() / na - (total - pooled[list(s)].sum()) / (n - na)
                      for s in combinations(range(n), na)])
    return float(obs), float(np.mean(np.abs(stats) >= abs(obs) - 1e-12)), len(stats)


def _welch_ci(a, b, level):
    from scipy import stats
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    half = stats.t.ppf(0.5 + level / 2, df) * np.sqrt(va + vb)
    diff = a.mean() - b.mean()
    return [float(diff - half), float(diff + half)]


def category_level(d, meta):
    """Category-level inference: categories (not items) are the exchangeable units.

    Uses the 21 category means of d. Exact permutation tests (I: 11 harm vs 5 suffering
    categories, 4368 splits; Delta_harm: 11 vs 5 neutral, 4368; Delta_suffer: 5 vs 5, 252)
    and Welch t intervals. Point estimates equal the item-level ones (equal n per category).
    Stays valid if the label effect varies by category, which the item-level tests do not.
    """
    cm = pd.Series(d, index=meta.index).groupby(meta["category"]).mean()
    grp = meta.groupby("category")["group"].first().loc[cm.index].to_numpy()
    v = {g: cm.to_numpy()[grp == g] for g in GROUPS}
    out = {}
    for name, a, b in (("I", "user_suffering", "harm_to_model"), ("delta_harm", "harm_to_model", "neutral"),
                       ("delta_suffer", "user_suffering", "neutral")):
        est, p, n = _exact_perm(v[a], v[b])
        out[name] = {"est": est, "p_exact_two_sided": p, "n_splits": n, "n_categories": [len(v[a]), len(v[b])],
                     "welch_ci95": _welch_ci(v[a], v[b], 0.95)}
        if name == "I":
            out[name]["welch_ci90"] = _welch_ci(v[a], v[b], 0.90)
    return out


def classify(I_est, I_p, I_ci90, harm_est, harm_sig, suffer_est, suffer_sig, sesoi, alpha=0.05):
    """Preregistered verdict. Combines the test of I with an equivalence test (TOST via the 90% CI)."""
    equiv = sesoi is not None and -sesoi < I_ci90[0] and I_ci90[1] < sesoi
    if I_p < alpha and equiv:
        return "trivial_interaction", f"Reliable interaction, but its 90% CI lies inside +/-{sesoi}: too small to matter."
    if I_p < alpha and I_est > 0:
        up, down = suffer_sig and suffer_est > 0, harm_sig and harm_est < 0
        if up and down:
            return "crossover", "Supports H-speaker: user suffering rises and harm-to-model falls, relative to neutral."
        if up:
            return "partial_suffer_only", "Partial: user suffering rises relative to neutral; harm-to-model does not reliably fall."
        if down:
            return "partial_harm_only", "Partial: harm-to-model falls relative to neutral; user suffering does not reliably rise."
        return "interaction_only", "Interaction in the H-speaker direction, neither component individually reliable."
    if I_p < alpha and I_est < 0:
        return "reverse_interaction", "Interaction opposite to H-speaker's prediction."
    if equiv:
        return "equivalent_parallel", (f"No interaction larger than +/-{sesoi} (90% CI inside bounds): consistent with "
                                       "H-model or a content confound, not with H-speaker.")
    return "inconclusive", "No reliable interaction, and equivalence to zero not established."


HSPEAKER_DIRECTION = ("crossover", "partial_suffer_only", "partial_harm_only", "interaction_only")


def combine(v_cat, v_item):
    """Co-primary verdict (conjunctive): a claim is made only as far as both inference levels support it.

    Same verdict -> that verdict. Both in the H-speaker direction -> the strongest verdict both
    support (crossover only if both say crossover; a partial verdict only if both allow it;
    otherwise 'interaction_only'). Anything else -> 'discordant' (no confirmatory claim).
    """
    a, b = v_cat[0], v_item[0]
    if a == b:
        return v_cat
    if a in HSPEAKER_DIRECTION and b in HSPEAKER_DIRECTION:
        for partial in ("partial_suffer_only", "partial_harm_only"):
            if {a, b} <= {"crossover", partial}:
                return partial, f"Both levels support at least '{partial}' (category: {a}; item: {b})."
        return "interaction_only", f"Both levels show an interaction in the H-speaker direction (category: {a}; item: {b})."
    return "discordant", f"Category level: {a}; item level: {b}. No confirmatory claim."


# ---------------------------------------------------------------- figures --------
def _style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(INK2)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def interaction_panel(ax, levels, outcome, conds, title, label_lines=True):
    x = np.arange(len(conds))
    for g in GROUPS:
        r = levels[(levels["outcome"] == outcome) & (levels["group"] == g)].set_index("condition").loc[conds]
        err = np.vstack([r["est"] - r["ci_lo"], r["ci_hi"] - r["est"]])
        ax.errorbar(x, r["est"], yerr=err, color=COLORS[g], marker=MARKERS[g], ms=7, lw=1.6,
                    elinewidth=1.2, capsize=0, mec=SURFACE, mew=1.2, label=GROUP_LABEL[g])
        if label_lines:
            ax.annotate(GROUP_LABEL[g], (x[-1], r["est"].iloc[-1]), xytext=(8, 0), textcoords="offset points",
                        va="center", fontsize=8.5, color=INK)
    ax.set_xticks(x)
    if label_lines:
        ax.set_xticklabels([f"{COND_LABEL[c]}\nnext" for c in conds], fontsize=9, color=INK)
    else:
        ax.set_xticklabels([COND_SHORT[c] for c in conds], fontsize=8.5, color=INK)
    ax.set_xlim(-0.3, len(conds) - 1 + (0.9 if label_lines else 0.3))
    ax.set_title(title, loc="left", fontsize=10.5, color=INK)
    _style(ax)


def save(fig, path):
    fig.savefig(path.with_suffix(".png"), dpi=200, facecolor=SURFACE)
    fig.savefig(path.with_suffix(".pdf"), facecolor=SURFACE)


def make_figures(out, levels, percat, conds, model):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # Fig 1: primary interaction plot.
    fig, ax = plt.subplots(figsize=(6.4, 4.2), facecolor=SURFACE)
    interaction_panel(ax, levels, PRIMARY, conds, f"{model}: pain axis by final speaker label")
    ax.set_ylabel("Pain axis (z, fixed to [Assistant]: pool)\nmean with 95% bootstrap CI", fontsize=9, color=INK2)
    fig.tight_layout()
    save(fig, out / "fig1_interaction_pain")
    plt.close(fig)

    # Fig 2: per-category label effect, in the style of the paper's Figure 6.
    prim = percat[(percat["outcome"] == PRIMARY) & (percat["condition"] == "user_next")]
    sizes = [int((prim["group"] == g).sum()) for g in GROUPS]
    fig, axes = plt.subplots(3, 1, figsize=(7.2, 0.3 * sum(sizes) + 2.6), sharex=True, facecolor=SURFACE,
                             gridspec_kw={"height_ratios": [s + 1 for s in sizes]})
    neutral_ref = percat.attrs.get("neutral_mean_d_user_next")
    for ax, g in zip(axes, GROUPS):
        sub = percat[(percat["group"] == g) & (percat["outcome"] == PRIMARY)]
        u = sub[sub["condition"] == "user_next"].sort_values("est")
        y = np.arange(len(u))
        ax.errorbar(u["est"], y, xerr=[u["est"] - u["ci_lo"], u["ci_hi"] - u["est"]], fmt=MARKERS[g],
                    color=COLORS[g], ms=7, elinewidth=1.2, capsize=0, mec=SURFACE, mew=1.2,
                    label="[User]: next minus [Assistant]: next")
        if "moderator_next" in conds:
            m = sub[sub["condition"] == "moderator_next"].set_index("category").loc[u["category"]]
            ax.plot(m["est"], y, MARKERS[g], ms=6, mfc="none", mec=COLORS[g], mew=1.2,
                    label="[Moderator]: next minus [Assistant]: next")
        ax.axvline(0, color=INK2, lw=1)
        if neutral_ref is not None:
            ax.axvline(neutral_ref, color=INK2, lw=1, ls=(0, (3, 3)))
        ax.set_yticks(y)
        ax.set_yticklabels(u["category"].str.replace("_", " "), fontsize=9, color=INK)
        ax.set_title(GROUP_LABEL[g], loc="left", fontsize=10.5, color=INK)
        _style(ax)
        ax.grid(axis="y", visible=False)
        ax.grid(axis="x", color=GRID, linewidth=0.8)
    from matplotlib.lines import Line2D
    handles = [Line2D([], [], marker="o", ls="none", color=INK2, ms=7, label="[User]: next minus [Assistant]: next")]
    if "moderator_next" in conds:
        handles.append(Line2D([], [], marker="o", ls="none", mfc="none", mec=INK2, mew=1.2, ms=6,
                              label="[Moderator]: next minus [Assistant]: next"))
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.01, 0.995), ncol=2, frameon=False, fontsize=8.5)
    axes[-1].set_xlabel("Label effect d on the pain axis (z): category mean, 95% bootstrap CI.\n"
                        "Solid line: 0. Dashed line: neutral mean d for [User]: (the pure label effect).",
                        fontsize=9, color=INK2)
    fig.suptitle(f"{model}: change in pain axis when the final label is swapped", x=0.01, y=0.955,
                 ha="left", fontsize=11, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    save(fig, out / "fig2_per_category_d")
    plt.close(fig)

    # Fig 3: comparison axes as small multiples (shared y: all fixed-z units).
    for name, outs in (("fig3_interaction_fear_negemo_sadness",
                        [("z_fear", "Fear"), ("z_negemo", "Negative emotion"), ("z_sadness", "Sadness")]),
                       ("fig4_interaction_S1_S2", [("z_S1", "S1 pain vector"), ("z_S2", "S2 pain vector")])):
        fig, axes = plt.subplots(1, len(outs), figsize=(2.9 * len(outs) + 1.0, 4.0), sharey=True, facecolor=SURFACE)
        for i, (ax, (col, lab)) in enumerate(zip(axes, outs)):
            interaction_panel(ax, levels, col, conds, lab, label_lines=False)
        axes[0].set_ylabel("z (fixed to [Assistant]: pool)\nmean with 95% CI", fontsize=9, color=INK2)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.01, 0.99), ncol=3, frameon=False, fontsize=8.5)
        fig.suptitle(f"{model}: comparison axes by final speaker label (x: label before the readout)",
                     x=0.01, y=0.9, ha="left", fontsize=10.5, color=INK)
        fig.tight_layout(rect=(0, 0, 1, 0.84))
        save(fig, out / name)
        plt.close(fig)


# ---------------------------------------------------------------- main -----------
def analyze(df, n_boot, n_perm, seed, sesoi):
    conds = [c for c in CONDS if c in set(df["condition"])]
    swapped = [c for c in conds if c != "assistant_next"]
    meta = (df[df["condition"] == "assistant_next"][["scenario_id", "category", "group", "n_user_turns"]]
            .sort_values("scenario_id").reset_index(drop=True))
    rng = np.random.default_rng(seed)
    design = Design(meta, n_boot, rng)

    def values(col, cond):
        w = df[df["condition"] == cond].set_index("scenario_id")[col]
        return w.loc[meta["scenario_id"]].to_numpy(dtype=float)

    contrast_rows, percat_rows, level_rows, summary = [], [], [], {"contrasts": {}}
    for col, desc, role in OUTCOMES:
        if col not in df:
            continue
        base = values(col, "assistant_next")
        for cond in conds:  # group levels per condition (paired resamples)
            lv = boot_summary(values(col, cond), design)
            for g in GROUPS:
                e = lv[f"mean_{g}"]
                level_rows.append({"outcome": col, "condition": cond, "group": g, "est": e["est"],
                                   "ci_lo": e["ci95"][0], "ci_hi": e["ci95"][1]})
        for cond in swapped:
            d = values(col, cond) - base
            res = boot_summary(d, design)
            i_obs, p = perm_test_I(d, design.groups, n_perm, np.random.default_rng(seed))
            assert abs(i_obs - res["I"]["est"]) < 1e-9
            res["I"]["p_perm_two_sided"] = p
            cat = category_level(d, meta)
            entry = {"outcome": col, "description": desc, "role": role, "condition": cond, **res,
                     "category_level": cat}
            if col == PRIMARY:
                c = cat
                entry["verdict_category"] = classify(
                    c["I"]["est"], c["I"]["p_exact_two_sided"], c["I"]["welch_ci90"],
                    c["delta_harm"]["est"], c["delta_harm"]["p_exact_two_sided"] < 0.05,
                    c["delta_suffer"]["est"], c["delta_suffer"]["p_exact_two_sided"] < 0.05, sesoi)
                entry["verdict_item"] = classify(
                    res["I"]["est"], p, res["I"]["ci90"],
                    res["delta_harm"]["est"], not (res["delta_harm"]["ci95"][0] <= 0 <= res["delta_harm"]["ci95"][1]),
                    res["delta_suffer"]["est"], not (res["delta_suffer"]["ci95"][0] <= 0 <= res["delta_suffer"]["ci95"][1]),
                    sesoi)
                entry["verdict_coprimary"] = combine(entry["verdict_category"], entry["verdict_item"])
            summary["contrasts"][f"{col}|{cond}"] = entry
            for k, v in res.items():
                contrast_rows.append({"outcome": col, "role": role, "condition": cond, "stat": k, "est": v["est"],
                                      "ci_lo": v["ci95"][0], "ci_hi": v["ci95"][1],
                                      "p_perm_item": v.get("p_perm_two_sided"),
                                      "p_exact_category": cat.get(k, {}).get("p_exact_two_sided"),
                                      "welch_ci95_lo": cat.get(k, {}).get("welch_ci95", [None])[0],
                                      "welch_ci95_hi": cat.get(k, {}).get("welch_ci95", [None, None])[1]})
            for cat, sub in meta.groupby("category"):
                ix = sub.index.to_numpy()
                bs = d[design.boot[:, ix]].mean(1)
                percat_rows.append({"outcome": col, "condition": cond, "group": sub["group"].iloc[0], "category": cat,
                                    "n": len(ix), "est": float(d[ix].mean()),
                                    "ci_lo": float(np.percentile(bs, 2.5)), "ci_hi": float(np.percentile(bs, 97.5))})

    # Multi-turn vs single-turn items (descriptive; no neutral items are multi-turn, so only I).
    summary["multi_turn"] = {}
    for cond in swapped:
        d = values(PRIMARY, cond) - values(PRIMARY, "assistant_next")
        for name, sel in (("multi_turn", meta["n_user_turns"] > 1), ("single_turn", meta["n_user_turns"] == 1)):
            m2 = meta[sel]
            sub_design = Design(m2, n_boot, np.random.default_rng(seed))
            dd = d[sel.to_numpy()]
            r = boot_summary(dd, sub_design)
            summary["multi_turn"][f"{name}|{cond}"] = {
                "n_harm": int((m2["group"] == "harm_to_model").sum()),
                "n_suffer": int((m2["group"] == "user_suffering").sum()),
                "n_neutral": int((m2["group"] == "neutral").sum()),
                "I": r["I"], "mean_harm": r["mean_harm_to_model"], "mean_suffer": r["mean_user_suffering"]}

    levels = pd.DataFrame(level_rows)
    percat = pd.DataFrame(percat_rows)
    if "user_next" in swapped:
        percat.attrs["neutral_mean_d_user_next"] = summary["contrasts"][f"{PRIMARY}|user_next"]["mean_neutral"]["est"]
    return summary, pd.DataFrame(contrast_rows), percat, levels, conds


def report(summary, model, sesoi):
    lines = [f"# Phase 4 analysis: {model}", ""]
    f = lambda e: f"{e['est']:+.3f} [{e['ci95'][0]:+.3f}, {e['ci95'][1]:+.3f}]"
    w = lambda e: f"{e['est']:+.3f} [{e['welch_ci95'][0]:+.3f}, {e['welch_ci95'][1]:+.3f}]"
    for cond in summary["conditions"][1:]:
        p = summary["contrasts"][f"{PRIMARY}|{cond}"]
        c = p["category_level"]
        lines += [f"## Pain axis, {COND_LABEL[cond]} vs [Assistant]:", "",
                  f"- **Co-primary verdict (both levels must agree): {p['verdict_coprimary'][0]}.** {p['verdict_coprimary'][1]}",
                  f"- Category-level verdict: {p['verdict_category'][0]}. {p['verdict_category'][1]}",
                  f"- Item-level verdict: {p['verdict_item'][0]}. {p['verdict_item'][1]}", "",
                  "| statistic | item level: est [95% bootstrap CI] | item perm p | category level: est [95% Welch CI] | exact p |",
                  "|---|---|---|---|---|",
                  f"| mean d, harm to model | {f(p['mean_harm_to_model'])} | | | |",
                  f"| mean d, user suffering | {f(p['mean_user_suffering'])} | | | |",
                  f"| mean d, neutral (label effect) | {f(p['mean_neutral'])} | | | |",
                  f"| Delta_harm | {f(p['delta_harm'])} | | {w(c['delta_harm'])} | {c['delta_harm']['p_exact_two_sided']:.4f} |",
                  f"| Delta_suffer | {f(p['delta_suffer'])} | | {w(c['delta_suffer'])} | {c['delta_suffer']['p_exact_two_sided']:.4f} |",
                  f"| I | {f(p['I'])} | {p['I']['p_perm_two_sided']:.4f} | {w(c['I'])} | {c['I']['p_exact_two_sided']:.4f} |",
                  f"| I, 90% CI (equivalence, SESOI +/-{sesoi}) | [{p['I']['ci90'][0]:+.3f}, {p['I']['ci90'][1]:+.3f}] | | "
                  f"[{c['I']['welch_ci90'][0]:+.3f}, {c['I']['welch_ci90'][1]:+.3f}] | |", ""]
    lines += ["## All outcomes: I = Delta_suffer - Delta_harm", "",
              "| outcome | condition | I, item [95% boot CI] | item perm p | I, category [95% Welch CI] | exact p |",
              "|---|---|---|---|---|---|"]
    for e in summary["contrasts"].values():
        c = e["category_level"]["I"]
        lines.append(f"| {e['outcome']} | {e['condition']} | {f(e['I'])} | {e['I']['p_perm_two_sided']:.4f} | "
                     f"{w(c)} | {c['p_exact_two_sided']:.4f} |")
    lines += ["", "## Multi-turn vs single-turn items (descriptive, pain axis, item-level bootstrap)", "",
              "| subset | condition | n harm / suffer | I [95% CI] |", "|---|---|---|---|"]
    for k, e in summary["multi_turn"].items():
        s, c = k.split("|")
        lines.append(f"| {s} | {c} | {e['n_harm']} / {e['n_suffer']} | {f(e['I'])} |")
    return "\n".join(lines) + "\n"


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--items", required=True, help="items.csv from 03_run_conditions.py")
    ap.add_argument("--out-dir", default=None, help="default: <items dir>/analysis_<timestamp>")
    ap.add_argument("--n-boot", type=int, default=10000)
    ap.add_argument("--n-perm", type=int, default=10000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--sesoi", type=float, default=0.25, help="equivalence bound for I, pain-axis z units")
    ap.add_argument("--no-figures", action="store_true")
    args = ap.parse_args()

    items = Path(args.items)
    df = pd.read_csv(items)
    model = df["model"].iloc[0]
    out = Path(args.out_dir) if args.out_dir else items.parent / f"analysis_{time.strftime('%Y%m%d_%H%M%S')}"
    if out.exists():
        raise SystemExit(f"{out} exists; refusing to overwrite")
    out.mkdir(parents=True)

    summary, contrasts_df, percat, levels, conds = analyze(df, args.n_boot, args.n_perm, args.seed, args.sesoi)
    summary.update({"model": model, "items": str(items), "n_boot": args.n_boot, "n_perm": args.n_perm,
                    "seed": args.seed, "sesoi": args.sesoi, "conditions": conds})
    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    contrasts_df.to_csv(out / "contrasts.csv", index=False)
    percat.to_csv(out / "per_category.csv", index=False)
    levels.to_csv(out / "levels.csv", index=False)
    text = report(summary, model, args.sesoi)
    (out / "REPORT_analysis.md").write_text(text)
    if not args.no_figures:
        make_figures(out, levels, percat, conds, model)
    print(text)
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
# Integrity check against the preregistration (normalizes the trailing newline written by %%writefile).
import hashlib
EXPECTED = {
 "pa_common.py": "3dc05ab285fd72516f3cc106dcbeae37c02b1fd789735b32c6fd4de1e9b90e7b",
 "02_build_stimuli.py": "16b1a724c8d0495aefee62de4f0d7247ac0963fc90dc53735a12053511aeeb13",
 "03_run_conditions.py": "a745b71b2193d85bfa2458f5ac33cb0a7eccc99fcb4a0014555bd2a81b8e3905",
 "04_analyze.py": "681ab37b130a35e7c67da649d34b1961b5966cbf1f49aaa9c5dc508a9fdab3b3"
}
for name, sha in EXPECTED.items():
    text = open(f"/content/speaker_study_scripts/{name}").read()
    got = hashlib.sha256((text.rstrip("\n") + "\n").encode()).hexdigest()
    assert got == sha, f"{name}: hash mismatch; do not run, tell Claude"
    print("ok", name, sha[:12])

In [ ]:
# Stimuli: rebuild from the paper's dataset (or confirm the copy on Drive) and check the frozen hash.
%cd /content/speaker_study_scripts
!python 02_build_stimuli.py --pain-axis-dir /content/Pain-axis --out-dir "{OUT_ROOT}/stimuli"
import hashlib
got = hashlib.sha256(open(f"{OUT_ROOT}/stimuli/stimuli.jsonl", "rb").read()).hexdigest()
assert got == "9d3b84ca8e32e20d07fea37a2884119739833abe711b0c242e039ed76b03d209", "stimuli.jsonl hash mismatch; do not run, tell Claude"
print("ok stimuli.jsonl", got[:12])

In [ ]:
# Phase 3 + 4, PRIMARY model: Qwen 2.5 7B base
%cd /content/speaker_study_scripts
!python 03_run_conditions.py --pain-axis-dir /content/Pain-axis --stimuli "{OUT_ROOT}/stimuli/stimuli.jsonl" --out-root "{OUT_ROOT}" --model-repo Qwen/Qwen2.5-7B --dtype bf16
import glob
run = sorted(glob.glob(f"{OUT_ROOT}/results/Qwen_2.5_7B_base/phase3_*"))[-1]
print("analysing", run)
!python 04_analyze.py --items "{run}/items.csv"

In [ ]:
# Phase 3 + 4, secondary model: Gemma 2 2B base
%cd /content/speaker_study_scripts
!python 03_run_conditions.py --pain-axis-dir /content/Pain-axis --stimuli "{OUT_ROOT}/stimuli/stimuli.jsonl" --out-root "{OUT_ROOT}" --model-repo google/gemma-2-2b --dtype bf16
import glob
run = sorted(glob.glob(f"{OUT_ROOT}/results/Gemma_2_2B_base/phase3_*"))[-1]
print("analysing", run)
!python 04_analyze.py --items "{run}/items.csv"

In [ ]:
# Bundle the latest Phase 3 run of each model (items, run info, analysis, figures) for Claude.
import glob, os, zipfile
paths = []
for model in ["Qwen_2.5_7B_base", "Gemma_2_2B_base"]:
    runs = sorted(glob.glob(f"{OUT_ROOT}/results/{model}/phase3_*"))
    if runs:
        paths += [p for p in glob.glob(runs[-1] + "/**", recursive=True) if os.path.isfile(p)]
zip_path = f"{OUT_ROOT}/speaker_study_phase3.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.relpath(p, OUT_ROOT))
print("\n".join(os.path.relpath(p, OUT_ROOT) for p in paths))
from google.colab import files
files.download(zip_path)